<a href="https://colab.research.google.com/github/Ducks-200/VanTuksapp/blob/main/Vantuks_Bets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os

# Ensure the current directory is in sys.path for module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from users import create_user, login_user
from wallet import show_balance, deposit, withdraw
from games import play_game_menu
from referrals import generate_referral

def main():
    print("\n🎰 WELCOME TO VANTUKS-BETS 🎰")
    print("PLAY. BET. WIN BIG.\n")

    user = None

    while True:
        print("\n1. Register")
        print("2. Login")
        print("3. Exit")

        choice = input("Select option: ")

        if choice == "1":
            create_user()

        elif choice == "2":
            user = login_user()
            if user:
                dashboard(user)

        elif choice == "3":
            print("Goodbye!")
            break

def dashboard(user):
    while True:
        print(f"\n👤 Welcome {user['username']}")
        print("1. Wallet")
        print("2. Games")
        print("3. Referral Link")
        print("4. Logout")

        choice = input("Select: ")

        if choice == "1":
            wallet_menu(user)

        elif choice == "2":
            play_game_menu(user)

        elif choice == "3":
            generate_referral(user)

        elif choice == "4":
            break

def wallet_menu(user):
    while True:
        print("\n💰 WALLET")
        print("1. View Balance")
        print("2. Deposit")
        print("3. Withdraw")
        print("4. Back")

        choice = input("Select: ")

        if choice == "1":
            show_balance(user)

        elif choice == "2":
            deposit(user)

        elif choice == "3":
            withdraw(user)

        elif choice == "4":
            break

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'users'

### Re-creating Application Modules

To resolve the `ModuleNotFoundError`, we need to ensure that all the Python module files (`database.py`, `users.py`, `wallet.py`, `games.py`, `referrals.py`) are actually created in the Colab environment. The previous `%%writefile` commands might not have been executed or their output was not persisted.

Let's re-run all the `%%writefile` cells to ensure these files are generated, and then re-add the current directory to `sys.path` to make them discoverable by the Python interpreter.

In [ ]:
%%writefile database.py

import json

DB_FILE = 'vantuks_bets.json'

def load_data():
    try:
        with open(DB_FILE, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {'users': [], 'transactions': [], 'games': []}

def save_data(data):
    with open(DB_FILE, 'w') as f:
        json.dump(data, f, indent=4)

def get_all_users():
    data = load_data()
    return data.get('users', [])

def add_user(user):
    data = load_data()
    data['users'].append(user)
    save_data(data)

def update_user(updated_user):
    data = load_data()
    for i, user in enumerate(data['users']):
        if user['username'] == updated_user['username']:
            data['users'][i] = updated_user
            break
    save_data(data)

def get_user_by_username(username):
    data = load_data()
    for user in data['users']:
        if user['username'] == username:
            return user
    return None

Writing database.py


In [ ]:
%%writefile users.py

import getpass
from database import add_user, get_user_by_username, update_user

def create_user():
    print("\n--- Register ---")
    username = input("Enter username: ")
    if get_user_by_username(username):
        print("Username already exists. Please choose a different one.")
        return

    password = getpass.getpass("Enter password: ")
    confirm_password = getpass.getpass("Confirm password: ")

    if password != confirm_password:
        print("Passwords do not match. Please try again.")
        return

    user = {
        'username': username,
        'password': password, # In a real app, you'd hash this password!
        'balance': 0.0,
        'referral_code': None,
        'referred_by': None
    }
    add_user(user)
    print(f"User '{username}' registered successfully!")

def login_user():
    print("\n--- Login ---")
    username = input("Enter username: ")
    password = getpass.getpass("Enter password: ")

    user = get_user_by_username(username)

    if user and user['password'] == password:
        print(f"Welcome back, {username}!")
        return user
    else:
        print("Invalid username or password.")
        return None

Writing users.py


In [ ]:
%%writefile wallet.py

from database import update_user, get_user_by_username

def show_balance(user):
    print(f"\nYour current balance: ${user['balance']:.2f}")

def deposit(user):
    while True:
        try:
            amount = float(input("Enter amount to deposit: "))
            if amount <= 0:
                print("Deposit amount must be positive.")
            else:
                user['balance'] += amount
                update_user(user)
                print(f"Successfully deposited ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")

def withdraw(user):
    while True:
        try:
            amount = float(input("Enter amount to withdraw: "))
            if amount <= 0:
                print("Withdrawal amount must be positive.")
            elif amount > user['balance']:
                print("Insufficient funds.")
            else:
                user['balance'] -= amount
                update_user(user)
                print(f"Successfully withdrew ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")

Writing wallet.py


In [ ]:
%%writefile games.py

import random
from database import update_user

def play_game_menu(user):
    while True:
        print("\n🎮 GAMES")
        print("1. Coin Flip (Bet on Heads or Tails)")
        print("2. Back")

        choice = input("Select a game: ")

        if choice == "1":
            play_coin_flip(user)
        elif choice == "2":
            break
        else:
            print("Invalid choice. Please try again.")

def play_coin_flip(user):
    print("\n--- Coin Flip ---")
    while True:
        try:
            bet_amount = float(input(f"Enter your bet (current balance: ${user['balance']:.2f}): "))
            if bet_amount <= 0:
                print("Bet amount must be positive.")
            elif bet_amount > user['balance']:
                print("Insufficient funds.")
            else:
                break
        except ValueError:
            print("Invalid bet amount. Please enter a number.")

    while True:
        guess = input("Heads or Tails? (H/T): ").upper()
        if guess in ['H', 'T']:
            break
        else:
            print("Invalid guess. Please enter 'H' for Heads or 'T' for Tails.")

    result = random.choice(['H', 'T'])
    print(f"Flipping the coin... It's {('Heads' if result == 'H' else 'Tails')}!")

    if guess == result:
        user['balance'] += bet_amount
        print(f"Congratulations! You won ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")
    else:
        user['balance'] -= bet_amount
        print(f"Sorry, you lost ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")

    update_user(user)

Writing games.py


In [ ]:
%%writefile referrals.py

import uuid
from database import update_user

def generate_referral(user):
    if not user.get('referral_code'):
        user['referral_code'] = str(uuid.uuid4())[:8] # Generate a short unique code
        update_user(user)
        print(f"\nYour new referral code is: {user['referral_code']}")
    else:
        print(f"\nYour referral code is: {user['referral_code']}")

Writing referrals.py


In [ ]:
import sys
import os

# Add the current directory to sys.path to ensure modules are found
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"Current directory '{os.getcwd()}' added to sys.path.")

Current directory '/content' added to sys.path.


Now that the files should be created and the path updated, let's verify their existence.

In [ ]:
# Verify that the module files have been created
!ls

database.py  games.py  referrals.py  sample_data  users.py  wallet.py


After running the above cells, you should see `database.py`, `users.py`, `wallet.py`, `games.py`, and `referrals.py` in the output of the `!ls` command. Once confirmed, please re-run the main application cell (`659c4145`) to start the betting application.

First, let's create the `database.py` file. This module will handle the persistent storage of user data, including their username, password, balance, and referral information.

In [ ]:
%%writefile database.py

import json

DB_FILE = 'vantuks_bets.json'

def load_data():
    try:
        with open(DB_FILE, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {'users': [], 'transactions': [], 'games': []}

def save_data(data):
    with open(DB_FILE, 'w') as f:
        json.dump(data, f, indent=4)

def get_all_users():
    data = load_data()
    return data.get('users', [])

def add_user(user):
    data = load_data()
    data['users'].append(user)
    save_data(data)

def update_user(updated_user):
    data = load_data()
    for i, user in enumerate(data['users']):
        if user['username'] == updated_user['username']:
            data['users'][i] = updated_user
            break
    save_data(data)

def get_user_by_username(username):
    data = load_data()
    for user in data['users']:
        if user['username'] == username:
            return user
    return None


Writing database.py


Next, we'll create the `users.py` module. This module will contain functions for user registration and login, interacting with the `database.py` to store and retrieve user information.

In [ ]:
%%writefile users.py

import getpass
from database import add_user, get_user_by_username, update_user

def create_user():
    print("\n--- Register ---")
    username = input("Enter username: ")
    if get_user_by_username(username):
        print("Username already exists. Please choose a different one.")
        return

    password = getpass.getpass("Enter password: ")
    confirm_password = getpass.getpass("Confirm password: ")

    if password != confirm_password:
        print("Passwords do not match. Please try again.")
        return

    user = {
        'username': username,
        'password': password, # In a real app, you'd hash this password!
        'balance': 0.0,
        'referral_code': None,
        'referred_by': None
    }
    add_user(user)
    print(f"User '{username}' registered successfully!")

def login_user():
    print("\n--- Login ---")
    username = input("Enter username: ")
    password = getpass.getpass("Enter password: ")

    user = get_user_by_username(username)

    if user and user['password'] == password:
        print(f"Welcome back, {username}!")
        return user
    else:
        print("Invalid username or password.")
        return None


Writing users.py


Now, let's create the `wallet.py` module. This will handle all operations related to a user's balance, such as viewing, depositing, and withdrawing funds.

In [ ]:
%%writefile wallet.py

from database import update_user, get_user_by_username

def show_balance(user):
    print(f"\nYour current balance: ${user['balance']:.2f}")

def deposit(user):
    while True:
        try:
            amount = float(input("Enter amount to deposit: "))
            if amount <= 0:
                print("Deposit amount must be positive.")
            else:
                user['balance'] += amount
                update_user(user)
                print(f"Successfully deposited ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")

def withdraw(user):
    while True:
        try:
            amount = float(input("Enter amount to withdraw: "))
            if amount <= 0:
                print("Withdrawal amount must be positive.")
            elif amount > user['balance']:
                print("Insufficient funds.")
            else:
                user['balance'] -= amount
                update_user(user)
                print(f"Successfully withdrew ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")


Writing wallet.py


Next, let's create the `games.py` module. This will contain a simple game for users to play and bet with their balance.

In [ ]:
%%writefile games.py

import random
from database import update_user

def play_game_menu(user):
    while True:
        print("\n🎮 GAMES")
        print("1. Coin Flip (Bet on Heads or Tails)")
        print("2. Back")

        choice = input("Select a game: ")

        if choice == "1":
            play_coin_flip(user)
        elif choice == "2":
            break
        else:
            print("Invalid choice. Please try again.")

def play_coin_flip(user):
    print("\n--- Coin Flip ---")
    while True:
        try:
            bet_amount = float(input(f"Enter your bet (current balance: ${user['balance']:.2f}): "))
            if bet_amount <= 0:
                print("Bet amount must be positive.")
            elif bet_amount > user['balance']:
                print("Insufficient funds.")
            else:
                break
        except ValueError:
            print("Invalid bet amount. Please enter a number.")

    while True:
        guess = input("Heads or Tails? (H/T): ").upper()
        if guess in ['H', 'T']:
            break
        else:
            print("Invalid guess. Please enter 'H' for Heads or 'T' for Tails.")

    result = random.choice(['H', 'T'])
    print(f"Flipping the coin... It's {('Heads' if result == 'H' else 'Tails')}!")

    if guess == result:
        user['balance'] += bet_amount
        print(f"Congratulations! You won ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")
    else:
        user['balance'] -= bet_amount
        print(f"Sorry, you lost ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")

    update_user(user)


Writing games.py


Finally, we'll create the `referrals.py` module. This module will allow users to generate referral codes and potentially track referrals (though the tracking logic might be more complex in a full implementation).

In [ ]:
%%writefile referrals.py

import uuid
from database import update_user

def generate_referral(user):
    if not user.get('referral_code'):
        user['referral_code'] = str(uuid.uuid4())[:8] # Generate a short unique code
        update_user(user)
        print(f"\nYour new referral code is: {user['referral_code']}")
    else:
        print(f"\nYour referral code is: {user['referral_code']}")


Writing referrals.py


All the necessary modules have been created. You can now run the main application cell (`9YYX7bgFTnJf`) again to start the betting application. If you want to delete the `vantuks_bets.json` file to reset the data, you can run the following command in a new cell: `!rm vantuks_bets.json`.

In [ ]:
import sys
import os

# Add the current directory to sys.path to ensure modules are found
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"Current directory '{os.getcwd()}' added to sys.path.")

Current directory '/content' added to sys.path.


In [ ]:
# Verify that the module files have been created
!ls

In [ ]:
import sys
import os

# Ensure the current directory is in sys.path for module imports
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from users import create_user, login_user
from wallet import show_balance, deposit, withdraw
from games import play_game_menu
from referrals import generate_referral

def main():
    print("\n🎰 WELCOME TO VANTUKS-BETS 🎰")
    print("PLAY. BET. WIN BIG.\n")

    user = None

    while True:
        print("\n1. Register")
        print("2. Login")
        print("3. Exit")

        choice = input("Select option: ")

        if choice == "1":
            create_user()

        elif choice == "2":
            user = login_user()
            if user:
                dashboard(user)

        elif choice == "3":
            print("Goodbye!")
            break

def dashboard(user):
    while True:
        print(f"\n👤 Welcome {user['username']}")
        print("1. Wallet")
        print("2. Games")
        print("3. Referral Link")
        print("4. Logout")

        choice = input("Select: ")

        if choice == "1":
            wallet_menu(user)

        elif choice == "2":
            play_game_menu(user)

        elif choice == "3":
            generate_referral(user)

        elif choice == "4":
            break

def wallet_menu(user):
    while True:
        print("\n💰 WALLET")
        print("1. View Balance")
        print("2. Deposit")
        print("3. Withdraw")
        print("4. Back")

        choice = input("Select: ")

        if choice == "1":
            show_balance(user)

        elif choice == "2":
            deposit(user)

        elif choice == "3":
            withdraw(user)

        elif choice == "4":
            break

if __name__ == "__main__":
    main()


🎰 WELCOME TO VANTUKS-BETS 🎰
PLAY. BET. WIN BIG.


1. Register
2. Login
3. Exit


### Re-creating Application Modules

To make the application work, we need to ensure that all the Python module files (`database.py`, `users.py`, `wallet.py`, `games.py`, `referrals.py`) are present in the Colab environment. I will re-create them now.

In [12]:
%%writefile database.py

import json

DB_FILE = 'vantuks_bets.json'

def load_data():
    try:
        with open(DB_FILE, 'r') as f:
            return json.load(f)
    except FileNotFoundError:
        return {'users': [], 'transactions': [], 'games': []}

def save_data(data):
    with open(DB_FILE, 'w') as f:
        json.dump(data, f, indent=4)

def get_all_users():
    data = load_data()
    return data.get('users', [])

def add_user(user):
    data = load_data()
    data['users'].append(user)
    save_data(data)

def update_user(updated_user):
    data = load_data()
    for i, user in enumerate(data['users']):
        if user['username'] == updated_user['username']:
            data['users'][i] = updated_user
            break
    save_data(data)

def get_user_by_username(username):
    data = load_data()
    for user in data['users']:
        if user['username'] == username:
            return user
    return None

Overwriting database.py


In [14]:
%%writefile games.py

import random
from database import update_user

def play_game_menu(user):
    while True:
        print("\n🎮 GAMES")
        print("1. Coin Flip (Bet on Heads or Tails)")
        print("2. Back")

        choice = input("Select a game: ")

        if choice == "1":
            play_coin_flip(user)
        elif choice == "2":
            break
        else:
            print("Invalid choice. Please try again.")

def play_coin_flip(user):
    print("\n--- Coin Flip ---")
    while True:
        try:
            bet_amount = float(input(f"Enter your bet (current balance: ${user['balance']:.2f}): "))
            if bet_amount <= 0:
                print("Bet amount must be positive.")
            elif bet_amount > user['balance']:
                print("Insufficient funds.")
            else:
                break
        except ValueError:
            print("Invalid bet amount. Please enter a number.")

    while True:
        guess = input("Heads or Tails? (H/T): ").upper()
        if guess in ['H', 'T']:
            break
        else:
            print("Invalid guess. Please enter 'H' for Heads or 'T' for Tails.")

    result = random.choice(['H', 'T'])
    print(f"Flipping the coin... It's {('Heads' if result == 'H' else 'Tails')}!")

    if guess == result:
        user['balance'] += bet_amount
        print(f"Congratulations! You won ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")
    else:
        user['balance'] -= bet_amount
        print(f"Sorry, you lost ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")

    update_user(user)

Overwriting games.py


In [13]:
%%writefile users.py

import getpass
import hashlib
from database import add_user, get_user_by_username, update_user

def create_user():
    print("\n--- Register ---")
    username = input("Enter username: ")
    if get_user_by_username(username):
        print("Username already exists. Please choose a different one.")
        return

    password = getpass.getpass("Enter password: ")
    confirm_password = getpass.getpass("Confirm password: ")

    if password != confirm_password:
        print("Passwords do not match. Please try again.")
        return

    # Hash the password before storing
    hashed_password = hashlib.sha256(password.encode()).hexdigest()

    user = {
        'username': username,
        'password': hashed_password,
        'balance': 0.0,
        'referral_code': None,
        'referred_by': None
    }
    add_user(user)
    print(f"User '{username}' registered successfully!")

def login_user():
    print("\n--- Login ---")
    username = input("Enter username: ")
    password = getpass.getpass("Enter password: ")

    user = get_user_by_username(username)

    if user:
        # Hash the entered password for comparison
        entered_password_hash = hashlib.sha256(password.encode()).hexdigest()
        if user['password'] == entered_password_hash:
            print(f"Welcome back, {username}!")
            return user
    print("Invalid username or password.")
    return None

Overwriting users.py


In [4]:
%%writefile wallet.py

from database import update_user, get_user_by_username

def show_balance(user):
    print(f"\nYour current balance: ${user['balance']:.2f}")

def deposit(user):
    while True:
        try:
            amount = float(input("Enter amount to deposit: "))
            if amount <= 0:
                print("Deposit amount must be positive.")
            else:
                user['balance'] += amount
                update_user(user)
                print(f"Successfully deposited ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")

def withdraw(user):
    while True:
        try:
            amount = float(input("Enter amount to withdraw: "))
            if amount <= 0:
                print("Withdrawal amount must be positive.")
            elif amount > user['balance']:
                print("Insufficient funds.")
            else:
                user['balance'] -= amount
                update_user(user)
                print(f"Successfully withdrew ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")

Writing wallet.py


In [5]:
%%writefile games.py

import random
from database import update_user

def play_game_menu(user):
    while True:
        print("\n🎮 GAMES")
        print("1. Coin Flip (Bet on Heads or Tails)")
        print("2. Back")

        choice = input("Select a game: ")

        if choice == "1":
            play_coin_flip(user)
        elif choice == "2":
            break
        else:
            print("Invalid choice. Please try again.")

def play_coin_flip(user):
    print("\n--- Coin Flip ---")
    while True:
        try:
            bet_amount = float(input(f"Enter your bet (current balance: ${user['balance']:.2f}): "))
            if bet_amount <= 0:
                print("Bet amount must be positive.")
            elif bet_amount > user['balance']:
                print("Insufficient funds.")
            else:
                break
        except ValueError:
            print("Invalid bet amount. Please enter a number.")

    while True:
        guess = input("Heads or Tails? (H/T): ").upper()
        if guess in ['H', 'T']:
            break
        else:
            print("Invalid guess. Please enter 'H' for Heads or 'T' for Tails.")

    result = random.choice(['H', 'T'])
    print(f"Flipping the coin... It's {('Heads' if result == 'H' else 'Tails')}!")

    if guess == result:
        user['balance'] += bet_amount
        print(f"Congratulations! You won ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")
    else:
        user['balance'] -= bet_amount
        print(f"Sorry, you lost ${bet_amount:.2f}. New balance: ${user['balance']:.2f}")

    update_user(user)

Writing games.py


In [6]:
%%writefile referrals.py

import uuid
from database import update_user

def generate_referral(user):
    if not user.get('referral_code'):
        user['referral_code'] = str(uuid.uuid4())[:8] # Generate a short unique code
        update_user(user)
        print(f"\nYour new referral code is: {user['referral_code']}")
    else:
        print(f"\nYour referral code is: {user['referral_code']}")

Writing referrals.py


In [7]:
import sys
import os

# Add the current directory to sys.path to ensure modules are found
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

print(f"Current directory '{os.getcwd()}' added to sys.path.")

Current directory '/content' added to sys.path.


In [8]:
# Verify that the module files have been created
!ls

database.py  games.py  referrals.py  sample_data  users.py  wallet.py


All the necessary modules have been recreated. You can now re-run the main application cell (`659c4145`) to start the betting application. If you want to delete the `vantuks_bets.json` file to reset the data, you can run the following command in a new cell: `!rm vantuks_bets.json`.

In [1]:
!ls

sample_data


In [15]:
%%bash
cat wallet.py


from database import update_user, get_user_by_username

def show_balance(user):
    print(f"\nYour current balance: ${user['balance']:.2f}")

def deposit(user):
    while True:
        try:
            amount = float(input("Enter amount to deposit: "))
            if amount <= 0:
                print("Deposit amount must be positive.")
            else:
                user['balance'] += amount
                update_user(user)
                print(f"Successfully deposited ${amount:.2f}. New balance: ${user['balance']:.2f}")
                break
        except ValueError:
            print("Invalid amount. Please enter a number.")

def withdraw(user):
    while True:
        try:
            amount = float(input("Enter amount to withdraw: "))
            if amount <= 0:
                print("Withdrawal amount must be positive.")
            elif amount > user['balance']:
                print("Insufficient funds.")
            else:
                user['balance'] -= amount
      

In [16]:
%%bash
cat referrals.py


import uuid
from database import update_user

def generate_referral(user):
    if not user.get('referral_code'):
        user['referral_code'] = str(uuid.uuid4())[:8] # Generate a short unique code
        update_user(user)
        print(f"\nYour new referral code is: {user['referral_code']}")
    else:
        print(f"\nYour referral code is: {user['referral_code']}")


In [ ]:
%%bash
cat vantuks_bets.json